<a href="https://colab.research.google.com/github/OmarAwadSaber/SupervisedLearningAssignment1/blob/main/Untitled6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif
from PCA_implementation import PCA

# 1. Loading the Breast Cancer dataset (standard numerical dataset)
print("--- Loading Numerical Dataset (Breast Cancer) ---")
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

# Split data into training and testing sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- EXPERIMENT 0: Baseline ---
# Using all original numerical features with Gaussian NB
# (Note: We still standardize for better NB performance)
mu_tr = np.mean(X_train, axis=0)
sd_tr = np.std(X_train, axis=0)
X_train_std = (X_train - mu_tr) / sd_tr
X_test_std = (X_test - mu_tr) / sd_tr

base_model = GaussianNB()
base_model.fit(X_train_std, y_train)
acc_baseline = accuracy_score(y_test, base_model.predict(X_test_std))
print(f"Baseline Accuracy: {acc_baseline:.4f}")

# --- EXPERIMENT A: Feature Selection ---
# Choosing top 10 features using ANOVA F-value
select = SelectKBest(score_func=f_classif, k=10)
X_train_fs = select.fit_transform(X_train_std, y_train)
X_test_fs = select.transform(X_test_std)

fs_model = GaussianNB().fit(X_train_fs, y_train)
acc_fs = accuracy_score(y_test, fs_model.predict(X_test_fs))
print(f"Feature Selection Accuracy: {acc_fs:.4f}")

# --- EXPERIMENT B: PCA (From Scratch) ---
# Reducing features down to k=5
k_val = 5
pca = PCA(n_components=k_val)
X_train_pca = pca.fit_transform(X_train)

# Transforming test data based ONLY on training stats
X_test_pca = pca.transform(X_test)

pca_model = GaussianNB().fit(X_train_pca, y_train)
acc_pca = accuracy_score(y_test, pca_model.predict(X_test_pca))
print(f"PCA Accuracy (k=5): {acc_pca:.4f}")

# =========================================================
# PART 4: VISUALIZATIONS
# =========================================================

# 1. Confusion Matrix for PCA Model
cm = confusion_matrix(y_test, pca_model.predict(X_test_pca))
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.title('Confusion Matrix (PCA Experiment)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# 2. Scree Plot (Cumulative Variance)
plt.figure(figsize=(7,5))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='s', color='darkred')
plt.title('Scree Plot: Explained Variance by Components')
plt.xlabel('Principal Component Index')
plt.ylabel('Cumulative Variance Explained')
plt.grid(True)
plt.show()

# 3. Bar Chart Comparison
plt.figure(figsize=(6,5))
labels = ['Baseline', 'Feat. Selection', 'PCA (k=5)']
vals = [acc_baseline, acc_fs, acc_pca]
plt.bar(labels, vals, color=['blue', 'orange', 'green'])
plt.ylim(0.8, 1.0) # Zooming in to see the differences
plt.title('Accuracy Comparison (Numerical Data)')
plt.show()